# Backpropagation and Initialization


Figures are from Prince, *[Understanding Deep Learning](https://udlbook.github.io/udlbook/)*
(UDL), unless another source is cited. This lecture covers UDL chapter 7.
Code cells adapted from the UDL notebooks (MIT license,
[github.com/udlbook/udlbook/tree/main/Notebooks](https://github.com/udlbook/udlbook/tree/main/Notebooks)).


## Outline

- Recap: the chain rule on a scalar network
- Backpropagation in matrix form
- Computational graphs and autograd
- Initialization: keeping the variance stable
- The Lottery Ticket Hypothesis

## Training needs the gradient of the loss

Loss: a sum over training examples (lecture 06 writes it as a mean; same algorithm, the $1/I$ folds into the learning rate)

$$
L[\boldsymbol{\phi}] = \sum_{i=1}^{I} \ell_i = \sum_{i=1}^{I} \mathrm{l}\bigl[\mathrm{f}[\mathbf{x}_i, \boldsymbol{\phi}], \mathbf{y}_i\bigr]
$$

SGD step on a batch $\mathcal{B}_t$:

$$
\boldsymbol{\phi}_{t+1} \longleftarrow \boldsymbol{\phi}_t - \alpha \sum_{i \in \mathcal{B}_t} \frac{\partial \ell_i[\boldsymbol{\phi}_t]}{\partial \boldsymbol{\phi}}
$$

For a deep network the parameters are all the biases and weights,

$$
\boldsymbol{\phi} = \{\boldsymbol{\beta}_0, \boldsymbol{\Omega}_0, \boldsymbol{\beta}_1, \boldsymbol{\Omega}_1, \ldots, \boldsymbol{\beta}_K, \boldsymbol{\Omega}_K\},
$$

so for **every example in the batch** we need

$$
\frac{\partial \ell_i}{\partial \boldsymbol{\beta}_k} \quad \text{and} \quad \frac{\partial \ell_i}{\partial \boldsymbol{\Omega}_k} \qquad \text{for every layer } k.
$$

## Why not just differentiate the network equation?

Model equation for one input, two hidden layers of three units each:

$$
\begin{aligned}
y' = \phi'_0 &+ \phi'_1 \mathrm{a}\left[\psi_{10} + \psi_{11}\mathrm{a}[\theta_{10} + \theta_{11}x] + \psi_{12}\mathrm{a}[\theta_{20} + \theta_{21}x] + \psi_{13}\mathrm{a}[\theta_{30} + \theta_{31}x]\right] \\
&+ \phi'_2 \mathrm{a}[\psi_{20} + \psi_{21}\mathrm{a}[\theta_{10} + \theta_{11}x] + \psi_{22}\mathrm{a}[\theta_{20} + \theta_{21}x] + \psi_{23}\mathrm{a}[\theta_{30} + \theta_{31}x]] \\
&+ \phi'_3 \underbrace{\mathrm{a}[\psi_{30} + \psi_{31}\underbrace{\mathrm{a}[\theta_{10} + \theta_{11}x]}_{\text{1st hidden layer}} + \psi_{32}\underbrace{\mathrm{a}[\theta_{20} + \theta_{21}x]}_{\text{1st hidden layer}} + \psi_{33}\underbrace{\mathrm{a}[\theta_{30} + \theta_{31}x]}_{\text{1st hidden layer}}]}_{\text{2nd hidden layer}}
\end{aligned}
$$

- Differentiating this by hand, once per parameter, repeats the same sub-expressions over and over.
- **Backpropagation** (Rumelhart, Hinton and Williams, [1986](https://www.nature.com/articles/323533a0)) organizes the work so that every sub-expression is computed once.


# Recap: the chain rule on a scalar network

Most of you have derived backpropagation for a one-unit-per-layer network
before, in DS701 or conceptually in DS340. This section compresses that
derivation to four slides so that the notation is fixed before we move to
matrices. If any of it is new, watch
[A3 Backprop by Hand (refresher)](A3-Backprop-by-Hand.qmd) first; it works
through every step.


## Toy network: one unit per layer

![One input, three hidden layers of one unit each, a least-squares loss against the label $y_i$.](figs/05-Backpropagation-and-Initialization/toy-network.png)

Write the network as a chain of **pre-activations** $f_k$ and **activations** $h_k$:

$$
\begin{aligned}
f_0 &= \beta_0 + \omega_0 \cdot x_i & \qquad f_2 &= \beta_2 + \omega_2 \cdot h_2 \\
h_1 &= \mathrm{a}[f_0] & h_3 &= \mathrm{a}[f_2] \\
f_1 &= \beta_1 + \omega_1 \cdot h_1 & f_3 &= \beta_3 + \omega_3 \cdot h_3 \\
h_2 &= \mathrm{a}[f_1] & \ell_i &= (f_3 - y_i)^2
\end{aligned}
$$

The **forward pass** evaluates these in order and stores every intermediate value.

## Backward pass: derivatives of the loss w.r.t. the intermediates

Start at the loss and apply the chain rule one link at a time, in reverse order:

$$
\begin{aligned}
\frac{\partial \ell_i}{\partial f_3} &= 2(f_3 - y_i) \\
\frac{\partial \ell_i}{\partial h_3} &= \frac{\partial f_3}{\partial h_3} \frac{\partial \ell_i}{\partial f_3} = \omega_3 \frac{\partial \ell_i}{\partial f_3}\\
\frac{\partial \ell_i}{\partial f_2} &= \frac{\partial h_3}{\partial f_2} \left( \frac{\partial f_3}{\partial h_3} \frac{\partial \ell_i}{\partial f_3} \right) = \mathbb{I}[f_2 > 0]\, \frac{\partial \ell_i}{\partial h_3}\\
\frac{\partial \ell_i}{\partial h_2} &= \frac{\partial f_2}{\partial h_2} \left( \frac{\partial h_3}{\partial f_2} \frac{\partial f_3}{\partial h_3} \frac{\partial \ell_i}{\partial f_3} \right) \\
&\;\;\vdots \\
\frac{\partial \ell_i}{\partial f_0} &= \frac{\partial h_1}{\partial f_0} \left( \frac{\partial f_1}{\partial h_1} \frac{\partial h_2}{\partial f_1} \frac{\partial f_2}{\partial h_2} \frac{\partial h_3}{\partial f_2} \frac{\partial f_3}{\partial h_3} \frac{\partial \ell_i}{\partial f_3} \right)
\end{aligned}
$$

![](figs/05-Backpropagation-and-Initialization/toy-backward-graph-edges.png)

Each line multiplies **one new local derivative** onto the product already computed for the layer closer to the loss. The term in parentheses is never recomputed.

## Backward pass: derivatives w.r.t. the parameters

Each parameter enters through exactly one pre-activation, $f_k = \beta_k + \omega_k h_k$, so one more chain-rule step:

$$
\begin{aligned}
\frac{\partial \ell_i}{\partial \omega_k} &= \frac{\partial f_k}{\partial \omega_k} \frac{\partial \ell_i}{\partial f_k} = h_k \frac{\partial \ell_i}{\partial f_k} \\[4pt]
\frac{\partial \ell_i}{\partial \beta_k} &= \frac{\partial f_k}{\partial \beta_k} \frac{\partial \ell_i}{\partial f_k} = \frac{\partial \ell_i}{\partial f_k}
\end{aligned}
$$

with $h_0 = x_i$.

The two local derivatives come from the two kinds of link:

- linear: $\dfrac{\partial f_k}{\partial h_k} = \omega_k$
- ReLU: $\dfrac{\partial h_{k+1}}{\partial f_k} = \dfrac{\partial \mathrm{ReLU}[f_k]}{\partial f_k} = \mathbb{I}[f_k > 0]$


![](figs/05-Backpropagation-and-Initialization/toy-parameter-derivative-graph.png)

![](figs/05-Backpropagation-and-Initialization/relu-derivative.png)


## Two things the scalar chain already tells us

1. **The forward pass must store the activations.** The derivative with respect to a weight is the upstream derivative *times the activation it multiplies*, $h_k\, \partial \ell_i / \partial f_k$. Double the activation, double the effect. So every partial derivative has to be evaluated **per input**, with that input's activations.
2. **The backward pass reuses work.** To know how a weight feeding layer $k$ changes the loss we need how $h_k$ changes $h_{k+1}$, how $h_{k+1}$ changes $h_{k+2}$, and so on to the loss. Everything after the first factor was already computed for layer $k+1$.


Both observations carry over unchanged to the matrix case. The only new question is **what shape each derivative has**.


# Backpropagation in matrix form

A real layer has a vector of hidden units and a matrix of weights, so the
derivatives in the chain become vectors and matrices. This section fixes the
shapes, then writes the complete forward and backward passes for a deep
network. This is the derivation to know for the quiz.


## The deep network

![](figs/05-Backpropagation-and-Initialization/deep-network-four-layers.png), Omega_1 in R^{2x4}, Omega_2 in R^{3x2}, Omega_3 in R^{2x3}." width="55%"}

$$
\begin{aligned}
\mathbf{h}_1 &= \mathbf{a}[\boldsymbol{\beta}_0 + \boldsymbol{\Omega}_0 \mathbf{x}] \\
\mathbf{h}_2 &= \mathbf{a}[\boldsymbol{\beta}_1 + \boldsymbol{\Omega}_1 \mathbf{h}_1] \\
\mathbf{h}_3 &= \mathbf{a}[\boldsymbol{\beta}_2 + \boldsymbol{\Omega}_2 \mathbf{h}_2] \\
\mathbf{f}[\mathbf{x}, \boldsymbol{\phi}] &= \boldsymbol{\beta}_3 + \boldsymbol{\Omega}_3 \mathbf{h}_3
\end{aligned}
$$

$\boldsymbol{\Omega}_k$ has one row per output unit and one column per input unit; $\mathbf{a}[\cdot]$ applies the activation function elementwise.

## Matrix calculus: derivative of a scalar

Scalar function $f[\cdot]$ of a *vector* $\mathbf{a}$: the derivative is a vector **of the shape of $\mathbf{a}$**.

$$
\mathbf{a} = \begin{bmatrix} a_1 \\ a_2 \\ a_3 \\ a_4 \end{bmatrix}
\qquad\qquad
\frac{\partial f}{\partial \mathbf{a}} = \begin{bmatrix} \frac{\partial f}{\partial a_1} \\ \frac{\partial f}{\partial a_2} \\ \frac{\partial f}{\partial a_3} \\ \frac{\partial f}{\partial a_4} \end{bmatrix}
$$

Scalar function $f[\cdot]$ of a *matrix* $\mathbf{A}$: the derivative is a matrix **of the shape of $\mathbf{A}$**.

$$
\mathbf{A} = \begin{bmatrix} a_{11} & a_{12} & a_{13} \\ a_{21} & a_{22} & a_{23} \\ a_{31} & a_{32} & a_{33} \\ a_{41} & a_{42} & a_{43} \end{bmatrix}
\qquad\qquad
\frac{\partial f}{\partial \mathbf{A}} = \begin{bmatrix}
\frac{\partial f}{\partial a_{11}} & \frac{\partial f}{\partial a_{12}} & \frac{\partial f}{\partial a_{13}} \\
\frac{\partial f}{\partial a_{21}} & \frac{\partial f}{\partial a_{22}} & \frac{\partial f}{\partial a_{23}} \\
\frac{\partial f}{\partial a_{31}} & \frac{\partial f}{\partial a_{32}} & \frac{\partial f}{\partial a_{33}} \\
\frac{\partial f}{\partial a_{41}} & \frac{\partial f}{\partial a_{42}} & \frac{\partial f}{\partial a_{43}}
\end{bmatrix}
$$

This is what makes the update $\boldsymbol{\Omega}_k \leftarrow \boldsymbol{\Omega}_k - \alpha\, \partial L / \partial \boldsymbol{\Omega}_k$ well defined.


## Matrix calculus: derivative of a vector

*Vector* function $\mathbf{f}[\cdot]$ of a *vector* $\mathbf{a}$:

$$
\mathbf{f} = \begin{bmatrix} f_1 \\ f_2 \\ f_3 \end{bmatrix}
\qquad
\mathbf{a} = \begin{bmatrix} a_1 \\ a_2 \\ a_3 \\ a_4 \end{bmatrix}
\qquad
\frac{\partial \mathbf{f}}{\partial \mathbf{a}} = \begin{bmatrix}
\frac{\partial f_1}{\partial a_1} & \frac{\partial f_2}{\partial a_1} & \frac{\partial f_3}{\partial a_1} \\
\frac{\partial f_1}{\partial a_2} & \frac{\partial f_2}{\partial a_2} & \frac{\partial f_3}{\partial a_2} \\
\frac{\partial f_1}{\partial a_3} & \frac{\partial f_2}{\partial a_3} & \frac{\partial f_3}{\partial a_3} \\
\frac{\partial f_1}{\partial a_4} & \frac{\partial f_2}{\partial a_4} & \frac{\partial f_3}{\partial a_4}
\end{bmatrix}
$$

- One **row** per input variable $a_j$, one **column** per output function $f_i$ (UDL's convention; it is the transpose of the Jacobian).
- With this convention the chain rule keeps the order of the scalar case: $\dfrac{\partial \ell}{\partial \mathbf{a}} = \dfrac{\partial \mathbf{f}}{\partial \mathbf{a}} \dfrac{\partial \ell}{\partial \mathbf{f}}$, and the result has the shape of $\mathbf{a}$.

## The two local derivatives, scalar vs. matrix

The linear link. Scalar:

$$
f_3 = \beta_3 + \omega_3 h_3
\qquad\qquad
\frac{\partial f_3}{\partial h_3} = \omega_3,
\qquad
\frac{\partial f_3}{\partial \beta_3} = 1
$$

Matrix:

$$
\mathbf{f}_3 = \boldsymbol{\beta}_3 + \boldsymbol{\Omega}_3 \mathbf{h}_3
\qquad\qquad
\frac{\partial \mathbf{f}_3}{\partial \mathbf{h}_3} = \boldsymbol{\Omega}_3^T,
\qquad
\frac{\partial \mathbf{f}_3}{\partial \boldsymbol{\beta}_3} = \mathbf{I}
$$

Check the shape: $\boldsymbol{\Omega}_3$ is (units of $\mathbf{f}_3$) $\times$ (units of $\mathbf{h}_3$), so $\boldsymbol{\Omega}_3^T$ has one row per element of $\mathbf{h}_3$, as the convention requires.


The ReLU link. Scalar: $\partial h_3 / \partial f_2 = \mathbb{I}[f_2 > 0]$. Matrix: a diagonal matrix with $\mathbb{I}[\mathbf{f}_2 > 0]$ on the diagonal, because unit $j$ of $\mathbf{h}_3$ depends only on unit $j$ of $\mathbf{f}_2$. Multiplying by it is an **elementwise product** $\mathbb{I}[\mathbf{f}_2 > 0] \odot (\cdot)$.


## Forward pass

![](figs/05-Backpropagation-and-Initialization/deep-network-intermediate-labels.png)

1. Write the network as a series of intermediate calculations.
2. Compute and **store** every intermediate quantity.


$$
\begin{aligned}
\mathbf{f}_0 &= \boldsymbol{\beta}_0 + \boldsymbol{\Omega}_0 \mathbf{x}_i \\
\mathbf{h}_1 &= \mathbf{a}[\mathbf{f}_0] \\
\mathbf{f}_1 &= \boldsymbol{\beta}_1 + \boldsymbol{\Omega}_1 \mathbf{h}_1 \\
\mathbf{h}_2 &= \mathbf{a}[\mathbf{f}_1] \\
\mathbf{f}_2 &= \boldsymbol{\beta}_2 + \boldsymbol{\Omega}_2 \mathbf{h}_2 \\
\mathbf{h}_3 &= \mathbf{a}[\mathbf{f}_2] \\
\mathbf{f}_3 &= \boldsymbol{\beta}_3 + \boldsymbol{\Omega}_3 \mathbf{h}_3 \\
\ell_i &= \mathrm{l}[\mathbf{f}_3, \mathbf{y}_i]
\end{aligned}
$$


## Backward pass: the chain of products

3. Take derivatives of the loss with respect to the intermediate quantities, in reverse order.

$$
\begin{aligned}
\mathbf{f}_0 &= \boldsymbol{\beta}_0 + \boldsymbol{\Omega}_0 \mathbf{x}_i \\
\mathbf{h}_1 &= \mathbf{a}[\mathbf{f}_0] \\
\mathbf{f}_1 &= \boldsymbol{\beta}_1 + \boldsymbol{\Omega}_1 \mathbf{h}_1 \\
\mathbf{h}_2 &= \mathbf{a}[\mathbf{f}_1] \\
\mathbf{f}_2 &= \boldsymbol{\beta}_2 + \boldsymbol{\Omega}_2 \mathbf{h}_2 \\
\mathbf{h}_3 &= \mathbf{a}[\mathbf{f}_2] \\
\mathbf{f}_3 &= \boldsymbol{\beta}_3 + \boldsymbol{\Omega}_3 \mathbf{h}_3 \\
\ell_i &= \mathrm{l}[\mathbf{f}_3, \mathbf{y}_i]
\end{aligned}
$$


$$
\begin{aligned}
&\frac{\partial \ell_i}{\partial \mathbf{f}_3} \qquad \text{(depends on the loss; e.g. } 2(\mathbf{f}_3 - \mathbf{y}_i)\text{)} \\
\frac{\partial \ell_i}{\partial \mathbf{f}_2} &= \frac{\partial \mathbf{h}_3}{\partial \mathbf{f}_2} \frac{\partial \mathbf{f}_3}{\partial \mathbf{h}_3} \frac{\partial \ell_i}{\partial \mathbf{f}_3} \\
\frac{\partial \ell_i}{\partial \mathbf{f}_1} &= \frac{\partial \mathbf{h}_2}{\partial \mathbf{f}_1} \frac{\partial \mathbf{f}_2}{\partial \mathbf{h}_2} \left( \frac{\partial \mathbf{h}_3}{\partial \mathbf{f}_2} \frac{\partial \mathbf{f}_3}{\partial \mathbf{h}_3} \frac{\partial \ell_i}{\partial \mathbf{f}_3} \right) \\
\frac{\partial \ell_i}{\partial \mathbf{f}_0} &= \frac{\partial \mathbf{h}_1}{\partial \mathbf{f}_0} \frac{\partial \mathbf{f}_1}{\partial \mathbf{h}_1} \left( \frac{\partial \mathbf{h}_2}{\partial \mathbf{f}_1} \frac{\partial \mathbf{f}_2}{\partial \mathbf{h}_2} \frac{\partial \mathbf{h}_3}{\partial \mathbf{f}_2} \frac{\partial \mathbf{f}_3}{\partial \mathbf{h}_3} \frac{\partial \ell_i}{\partial \mathbf{f}_3} \right)
\end{aligned}
$$

Same structure as the scalar case: each line adds two local derivatives (one ReLU, one linear) to a product that is already known.


## Backward pass: substituting the local derivatives

Take one line and substitute what we found for the two kinds of link:

$$
\frac{\partial \ell_i}{\partial \mathbf{f}_2} = \boxed{\frac{\partial \mathbf{h}_3}{\partial \mathbf{f}_2}} \boxed{\frac{\partial \mathbf{f}_3}{\partial \mathbf{h}_3}} \frac{\partial \ell_i}{\partial \mathbf{f}_3}
$$

$$
\frac{\partial \mathbf{f}_3}{\partial \mathbf{h}_3} = \boldsymbol{\Omega}_3^T
\qquad\qquad
\frac{\partial \mathbf{h}_3}{\partial \mathbf{f}_2} = \mathrm{diag}\bigl(\mathbb{I}[\mathbf{f}_2 > 0]\bigr)
$$


$$
\frac{\partial \ell_i}{\partial \mathbf{f}_2} = \mathbb{I}[\mathbf{f}_2 > 0] \odot \left( \boldsymbol{\Omega}_3^T \frac{\partial \ell_i}{\partial \mathbf{f}_3} \right)
$$

Read it right to left: **multiply by the transposed weights, then zero out the units whose ReLU was off.** The stored pre-activation $\mathbf{f}_2$ from the forward pass tells us which.


## Backward pass: derivatives w.r.t. the parameters

4. Take derivatives with respect to the parameters, via the pre-activation each one feeds.

Biases:

$$
\begin{aligned}
\frac{\partial \ell_i}{\partial \boldsymbol{\beta}_k} &= \frac{\partial \mathbf{f}_k}{\partial \boldsymbol{\beta}_k} \frac{\partial \ell_i}{\partial \mathbf{f}_k} \\
&= \frac{\partial}{\partial \boldsymbol{\beta}_k}\left(\boldsymbol{\beta}_k + \boldsymbol{\Omega}_k \mathbf{h}_k\right) \frac{\partial \ell_i}{\partial \mathbf{f}_k} \\
&= \frac{\partial \ell_i}{\partial \mathbf{f}_k}
\end{aligned}
$$


Weights:

$$
\begin{aligned}
\frac{\partial \ell_i}{\partial \boldsymbol{\Omega}_k} &= \frac{\partial \mathbf{f}_k}{\partial \boldsymbol{\Omega}_k} \frac{\partial \ell_i}{\partial \mathbf{f}_k} \\
&= \frac{\partial}{\partial \boldsymbol{\Omega}_k}\left(\boldsymbol{\beta}_k + \boldsymbol{\Omega}_k \mathbf{h}_k\right) \frac{\partial \ell_i}{\partial \mathbf{f}_k} \\
&= \frac{\partial \ell_i}{\partial \mathbf{f}_k} \mathbf{h}_k^T
\end{aligned}
$$


Shape check on the weights: $\partial \ell_i / \partial \mathbf{f}_k$ is a column with one entry per output unit, $\mathbf{h}_k^T$ a row with one entry per input unit; their **outer product** has the shape of $\boldsymbol{\Omega}_k$. Entry $(j, m)$ is (upstream derivative at output $j$) $\times$ (activation at input $m$): the scalar rule $h_k\, \partial \ell_i/\partial f_k$, once per weight.


## The complete algorithm (UDL section 7.4)

**Forward pass.** With $\mathbf{h}_0 = \mathbf{x}_i$, for $k = 0, \ldots, K$:

$$
\begin{aligned}
\mathbf{f}_k &= \boldsymbol{\beta}_k + \boldsymbol{\Omega}_k \mathbf{h}_k \\
\mathbf{h}_{k+1} &= \mathbf{a}[\mathbf{f}_k]
\end{aligned}
$$

Store every $\mathbf{f}_k$ and $\mathbf{h}_k$. The output is $\mathbf{f}_K$ and the loss $\ell_i = \mathrm{l}[\mathbf{f}_K, \mathbf{y}_i]$.


**Backward pass.** Start from $\partial \ell_i / \partial \mathbf{f}_K$. For $k = K, K-1, \ldots, 0$:

$$
\begin{aligned}
\frac{\partial \ell_i}{\partial \boldsymbol{\beta}_k} &= \frac{\partial \ell_i}{\partial \mathbf{f}_k} \\
\frac{\partial \ell_i}{\partial \boldsymbol{\Omega}_k} &= \frac{\partial \ell_i}{\partial \mathbf{f}_k} \mathbf{h}_k^T \\
\frac{\partial \ell_i}{\partial \mathbf{f}_{k-1}} &= \mathbb{I}[\mathbf{f}_{k-1} > 0] \odot \left( \boldsymbol{\Omega}_k^T \frac{\partial \ell_i}{\partial \mathbf{f}_k} \right)
\end{aligned}
$$

(the last line for $k \geq 1$).


Four operations per layer: one matrix-vector product, one outer product, one elementwise mask, one copy. Over a batch, the per-example gradients are summed, so the matrix-vector products become matrix-matrix products.


The algorithm in NumPy, on the network drawn earlier ($D_i = 3$, hidden
widths 4, 2, 3, $D_o = 2$) with a least-squares loss. `forward` stores every
$\mathbf{f}_k$ and $\mathbf{h}_k$; `backward` is the three lines of the box
above, one loop iteration per layer.

In [ ]:
#| code-fold: false
# Adapted from UDL notebook 7.2 (MIT)
import numpy as np

rng = np.random.default_rng(0)
dims = [3, 4, 2, 3, 2]          # D_i, three hidden widths, D_o
K = len(dims) - 2               # Omega_0 ... Omega_K
Omegas = [rng.normal(size=(dims[k + 1], dims[k])) for k in range(K + 1)]
betas = [rng.normal(size=(dims[k + 1], 1)) for k in range(K + 1)]
x = rng.normal(size=(dims[0], 1))
y = rng.normal(size=(dims[-1], 1))

def forward(x):
    fs, hs = [], [x]                                  # h_0 = x
    for k in range(K + 1):
        fs.append(betas[k] + Omegas[k] @ hs[k])       # f_k = beta_k + Omega_k h_k
        if k < K:
            hs.append(np.maximum(fs[k], 0))           # h_{k+1} = ReLU[f_k]
    return fs, hs

def loss(x, y):
    return np.sum((forward(x)[0][K] - y) ** 2)

def backward(fs, hs, y):
    dl_dOmegas, dl_dbetas = [None] * (K + 1), [None] * (K + 1)
    dl_df = 2 * (fs[K] - y)                           # dl/df_K for least squares
    for k in range(K, -1, -1):
        dl_dbetas[k] = dl_df.copy()                   # dl/dbeta_k = dl/df_k
        dl_dOmegas[k] = dl_df @ hs[k].T               # outer product with h_k
        if k > 0:
            dl_df = (fs[k - 1] > 0) * (Omegas[k].T @ dl_df)   # mask after Omega^T
    return dl_dOmegas, dl_dbetas

fs, hs = forward(x)
dl_dOmegas, dl_dbetas = backward(fs, hs, y)
for k in range(K):
    print(f"ReLU mask I[f_{k} > 0] = {(fs[k] > 0).astype(int).ravel()}")

Check every derivative against a central finite difference,
$(\ell[\phi + \epsilon] - \ell[\phi - \epsilon]) / 2\epsilon$, one parameter at
a time. That costs two forward passes **per parameter**, which is exactly why
nobody trains this way; backpropagation gets all of them from one backward
pass.

In [ ]:
#| code-fold: false
def finite_difference(param, eps=1e-6):
    grad = np.zeros_like(param)
    for idx in np.ndindex(param.shape):
        old = param[idx]
        param[idx] = old + eps
        up = loss(x, y)
        param[idx] = old - eps
        down = loss(x, y)
        param[idx] = old
        grad[idx] = (up - down) / (2 * eps)
    return grad

def rel_error(a, b):
    return np.max(np.abs(a - b)) / max(np.max(np.abs(a)), np.max(np.abs(b)), 1e-12)

errors = []
for k in range(K + 1):
    errors.append(rel_error(dl_dOmegas[k], finite_difference(Omegas[k])))
    errors.append(rel_error(dl_dbetas[k], finite_difference(betas[k])))
print(f"max relative error over all {sum(O.size + b.size for O, b in zip(Omegas, betas))} "
      f"parameters: {max(errors):.1e}")

An error around $10^{-9}$ is the finite-difference rounding error; a bug in
the backward pass (a missing transpose, the mask on the wrong layer) gives an
error of order 1. Try deleting the `(fs[k - 1] > 0) *` factor.


## Check it with numbers

One layer with two inputs and two outputs. From the forward pass we stored $\mathbf{f}_{k-1} = \begin{bmatrix} 1 \\ -1 \end{bmatrix}$, so $\mathbf{h}_k = \mathrm{ReLU}[\mathbf{f}_{k-1}] = \begin{bmatrix} 1 \\ 0 \end{bmatrix}$. The weights are $\boldsymbol{\Omega}_k = \begin{bmatrix} 2 & -1 \\ 0 & 3 \end{bmatrix}$, and the upstream derivative arriving from the layer above is $\dfrac{\partial \ell_i}{\partial \mathbf{f}_k} = \begin{bmatrix} 1 \\ 2 \end{bmatrix}$.

$$
\frac{\partial \ell_i}{\partial \boldsymbol{\beta}_k} = \begin{bmatrix} 1 \\ 2 \end{bmatrix}
\qquad
\frac{\partial \ell_i}{\partial \boldsymbol{\Omega}_k} = \begin{bmatrix} 1 \\ 2 \end{bmatrix} \begin{bmatrix} 1 & 0 \end{bmatrix} = \begin{bmatrix} 1 & 0 \\ 2 & 0 \end{bmatrix}
$$

The second column is zero: the second input unit was inactive, so its weights get no gradient from this example.


$$
\boldsymbol{\Omega}_k^T \frac{\partial \ell_i}{\partial \mathbf{f}_k} = \begin{bmatrix} 2 & 0 \\ -1 & 3 \end{bmatrix} \begin{bmatrix} 1 \\ 2 \end{bmatrix} = \begin{bmatrix} 2 \\ 5 \end{bmatrix}
$$

$$
\frac{\partial \ell_i}{\partial \mathbf{f}_{k-1}} = \begin{bmatrix} 1 \\ 0 \end{bmatrix} \odot \begin{bmatrix} 2 \\ 5 \end{bmatrix} = \begin{bmatrix} 2 \\ 0 \end{bmatrix}
$$

The mask also stops the gradient *flowing through* the inactive unit to the layers below.


## Cost of backpropagation

- **Compute.** The backward pass is about twice the forward pass: for each weight matrix the forward pass does one product ($\boldsymbol{\Omega}_k \mathbf{h}_k$) and the backward pass does two ($\boldsymbol{\Omega}_k^T\, \partial\ell/\partial\mathbf{f}_k$ and the outer product), except that the first layer can skip $\boldsymbol{\Omega}_0^T\, \partial\ell/\partial\mathbf{f}_0$, since no gradient with respect to the input is needed. This is the origin of the rule of thumb that training a transformer with $N$ parameters costs about $6N$ FLOPs per token, $2N$ forward and $4N$ backward ([Kaplan et al., 2020](https://arxiv.org/abs/2001.08361), section 2.1).
- **Memory.** Every $\mathbf{f}_k$ and $\mathbf{h}_k$ for every example in the batch must be kept until the backward pass reaches it. For large models the stored activations, not the parameters, dominate memory.
- **Sequential.** Layer $k$'s backward step cannot start before layer $k+1$'s is done. Examples in a batch are independent, so they run in parallel; splitting one model across machines is harder (lecture 12).
- Only matrix products and thresholding: exactly the operations GPUs are built for.

Counting it out: a product with an $m \times n$ weight matrix costs $mn$
multiplies and about $mn$ adds, $2mn$ FLOPs. The forward pass does one such
product per layer, the backward pass two (the cell counts both for every
layer, so for a shallow network it slightly overstates the backward cost: the
first layer's $\boldsymbol{\Omega}_0^T$ product is not needed). Change the widths, the parameter
count or the token budget below.

In [ ]:
widths = [3, 4, 2, 3, 2]                  # the network drawn earlier
weights = [widths[k] * widths[k + 1] for k in range(len(widths) - 1)]
N = sum(weights)                          # weight count (biases add only O(width) work)

fwd = sum(2 * w for w in weights)         # Omega_k h_k
bwd_outer = sum(2 * w for w in weights)   # (dl/df_k) h_k^T
bwd_input = sum(2 * w for w in weights)   # Omega_k^T dl/df_k (unneeded for k = 0)
print(f"N = {N} weights: forward {fwd} FLOPs = {fwd / N:.0f}N, "
      f"backward {bwd_outer + bwd_input} FLOPs = {(bwd_outer + bwd_input) / N:.0f}N, "
      f"total {(fwd + bwd_outer + bwd_input) / N:.0f}N")

N_params = 124e6                          # e.g. GPT-2 small
tokens = 10e9                             # training tokens
print(f"training a {N_params / 1e6:.0f}M-parameter model on {tokens / 1e9:.0f}B tokens: "
      f"6 * N * tokens = {6 * N_params * tokens:.2e} FLOPs")

# Computational graphs and autograd

We derived the backward pass for one architecture, the fully connected ReLU
network. Nobody re-derives it for every new architecture; the recipe above is
a special case of reverse-mode automatic differentiation, which a framework
applies to any composition of differentiable operations. This section is the
bridge between the derivation and the `loss.backward()` you will call in
every notebook.


## Backpropagation as a computational graph

Every forward computation is a **directed acyclic graph**: nodes are operations (matrix product, add, ReLU, loss), edges carry the tensors between them.

- The **forward pass** walks the graph from inputs to the loss, and each node stores what its backward step will need (the ReLU stores its mask, the product stores its inputs).
- The **backward pass** walks the same graph in reverse topological order. Each node receives $\partial \ell / \partial (\text{its output})$, multiplies by its own local derivative, and passes $\partial \ell / \partial (\text{its inputs})$ down the edges.
- When a tensor feeds **several** nodes (a residual connection, a weight used twice), the derivatives arriving along the different paths are **summed**. That is the multivariable chain rule.
- The two local derivatives we derived, $\boldsymbol{\Omega}^T(\cdot)$ and $\mathbb{I}[\mathbf{f} > 0] \odot (\cdot)$, are the backward rules of two node types. A framework ships a backward rule for every operation it offers.


## Reverse-mode automatic differentiation

Automatic differentiation computes exact derivatives of a program by applying the chain rule to its operations; it is neither symbolic differentiation nor finite differences ([Baydin et al., 2018](https://arxiv.org/abs/1502.05767)).

**Forward mode** pushes one input perturbation through the graph:

- one pass gives $\partial(\text{all outputs}) / \partial(\text{one input})$
- cost: one pass **per parameter**


**Reverse mode** pulls one output sensitivity back through the graph:

- one pass gives $\partial(\text{one output}) / \partial(\text{all inputs})$
- cost: one pass **per output**


Training has one scalar output (the loss) and millions or billions of inputs (the parameters). Reverse mode gets the whole gradient in a single backward pass. Backpropagation **is** reverse-mode AD applied to a neural network.


## What PyTorch autograd records

- A tensor created with `requires_grad=True` is a **leaf**: autograd will accumulate a gradient for it in `.grad`.
- Every operation on such a tensor produces a tensor with a `grad_fn`, the node that knows how to compute the operation's local derivative. The graph is built **dynamically**, as the forward code runs, so Python control flow (loops, `if`s) is fine.
- `loss.backward()` seeds $\partial \ell / \partial \ell = 1$ at the loss and runs the reverse traversal, **adding** each leaf's result into its `.grad`.
- Gradients accumulate across calls on purpose (gradient accumulation across micro-batches), so a training loop calls `optimizer.zero_grad()` before each step.
- `with torch.no_grad():` skips graph construction (inference, parameter updates); `.detach()` cuts a tensor out of the graph.

## The numeric check, in PyTorch

In [ ]:
#| eval: false
#| code-fold: false
import torch

f_prev = torch.tensor([1.0, -1.0])                  # stored pre-activation f_{k-1}
Omega = torch.tensor([[2.0, -1.0], [0.0, 3.0]], requires_grad=True)
beta = torch.zeros(2, requires_grad=True)
upstream = torch.tensor([1.0, 2.0])                 # dl/df_k from the layer above

h = torch.relu(f_prev.requires_grad_())             # h_k = [1, 0]
f = beta + Omega @ h                                # f_k
f.backward(gradient=upstream)                       # seed the backward pass with dl/df_k

print(Omega.grad)    # tensor([[1., 0.], [2., 0.]])
print(beta.grad)     # tensor([1., 2.])
print(f_prev.grad)   # tensor([2., 0.])

Normally you call `loss.backward()` on a scalar, which seeds the pass with 1.
Passing `gradient=` to `backward()` on a vector seeds it with an arbitrary
upstream derivative, which is exactly what lets us test one layer in
isolation. The three printed tensors match the hand calculation on the
previous slides.

The same three gradients by hand, in NumPy, using the backward-pass rules:

In [ ]:
#| code-fold: false
f_prev = np.array([1.0, -1.0])            # stored pre-activation f_{k-1}
Omega = np.array([[2.0, -1.0], [0.0, 3.0]])
upstream = np.array([1.0, 2.0])           # dl/df_k from the layer above

h = np.maximum(f_prev, 0)                 # h_k = [1, 0]
print("Omega.grad  =", np.outer(upstream, h).tolist())           # (dl/df_k) h_k^T
print("beta.grad   =", upstream.tolist())                        # dl/df_k
print("f_prev.grad =", ((f_prev > 0) * (Omega.T @ upstream)).tolist())  # I[f > 0] * Omega^T dl/df_k

## Autograd does not make memory free

- Everything the backward pass needs is kept alive from the forward pass: for a transformer that is every attention map and every MLP activation for every token in the batch.
- **Activation checkpointing** (also called gradient checkpointing or recomputation) stores only some layers' activations and **recomputes** the rest during the backward pass, trading about one extra forward pass for memory that grows as the square root of depth instead of linearly ([Chen et al., 2016](https://arxiv.org/abs/1604.06174)). In PyTorch: `torch.utils.checkpoint`. Every large-model training recipe uses it; we return to it in lecture 12.
- Mixed precision stores activations in 16-bit floats for the same reason: half the bytes per stored tensor.

- For another walk through autograd from scratch, see [Andrej Karpathy's micrograd tutorial](https://youtu.be/VMj-3S1tku0), which builds a scalar computational graph with `.backward()` in about 100 lines of Python.

# Initialization: keeping the variance stable

Gradient descent needs a starting point, and the backward pass we just derived
multiplies by a weight matrix at every layer. If those matrices are too large
or too small, the signal explodes or vanishes as it crosses the layers, in
both directions. This section works out what initial weight variance keeps it
stable, which is the He initialization (UDL section 7.5).


## Initialization: choosing the starting point

The standard building block, in terms of pre-activations:

$$
\begin{aligned}
\mathbf{f}_k &= \boldsymbol{\beta}_k + \boldsymbol{\Omega}_k \mathbf{h}_k \\
&= \boldsymbol{\beta}_k + \boldsymbol{\Omega}_k \mathbf{a}[\mathbf{f}_{k-1}]
\end{aligned}
$$

How do we initialize the biases and weights? Equivalent to choosing the starting point of the gradient-descent search.

The usual choice:

- all biases $\boldsymbol{\beta}_k = \mathbf{0}$
- weights normally distributed, mean 0, variance $\sigma^2_\Omega$


![Normal distributions with variance 0.5 and variance 100.](figs/05-Backpropagation-and-Initialization/normal-distributions-two-variances.png)


What happens to $\mathbf{f}_k$ as we move **forward** through the network if $\sigma^2_\Omega$ is very small? Very large?


## The same question for the backward pass

$$
\frac{\partial \ell_i}{\partial \mathbf{f}_{k-1}} = \mathbb{I}[\mathbf{f}_{k-1} > 0] \odot \left( \boldsymbol{\Omega}_k^T \frac{\partial \ell_i}{\partial \mathbf{f}_k} \right), \qquad k \in \{K, K-1, \ldots, 1\}
$$

- Each backward step multiplies by $\boldsymbol{\Omega}_k^T$. If the weights are very small, the gradient shrinks at every layer: **vanishing gradients**, and the early layers barely move.
- If the weights are very large, it grows at every layer: **exploding gradients**, and the early layers take wild steps.
- Same mechanism as the forward pass, with $\boldsymbol{\Omega}_k^T$ in place of $\boldsymbol{\Omega}_k$.


## Experiment: 50 layers, five weight variances

![UDL Fig. 7.7. Weight initialization. A deep network with 50 hidden layers and $D_h = 100$ hidden units per layer; 100-dimensional input $\mathbf{x}$ drawn from a standard normal, a single output fixed at $y = 0$, least-squares loss. The biases $\boldsymbol{\beta}_k$ are zero and the weights $\boldsymbol{\Omega}_k$ are normal with mean zero and five different variances $\sigma^2_\Omega \in \{0.001, 0.01, 0.02, 0.1, 1.0\}$.](figs/05-Backpropagation-and-Initialization/weight-init-variance-forward-backward.png)


- Input: 100D, $\sim N(0,1)$
- Large $\sigma^2_\Omega$ (1.0, 0.1): **exploding**
- Small $\sigma^2_\Omega$ (0.01, 0.001): **vanishing**
- $\sigma^2_\Omega = 0.02$: flat in **both** passes. Where does 0.02 come from?


The experiment, rerun: 1000 inputs pushed through 50 ReLU layers of width 100
with zero biases, then a least-squares loss against $y = 0$ pulled back
through the same layers with the backward pass from earlier. Each curve is
scaled to 1 at the layer where its pass starts. Dotted lines are the
prediction derived below, a factor of $D_h \sigma^2_\Omega / 2$ per layer.
Add your own $\sigma^2_\Omega$ to the list.

In [ ]:
#| code-fold: true
#| fig-cap: "Variance of the pre-activations (forward) and of their gradients (backward) at each of 50 layers of width 100, for five weight variances, each relative to the layer where its pass starts. Dotted: the predicted factor of D_h σ²_Ω / 2 per layer."
#| fig-alt: "Two log-scale line plots of variance, relative to the starting layer, against layer index 0 to 49. Left, forward pass: the lines for weight variance 1.0 and 0.1 rise steeply, 0.02 stays flat near 1, 0.01 and 0.001 fall steeply. Right, backward pass, read from layer 49 down to 0: the same pattern, with 0.02 flat. Dotted black prediction lines lie on top of the measured ones."
# Adapted from UDL notebook 7.3 (MIT)
import matplotlib.pyplot as plt

D_h, n_layers, n_data = 100, 50, 1000
sigma2_omegas = [0.001, 0.01, 0.02, 0.1, 1.0]

fig, (ax_f, ax_b) = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for sigma2 in sigma2_omegas:
    rng = np.random.default_rng(0)
    Omegas_init = [rng.normal(0, np.sqrt(sigma2), size=(D_h, D_h)) for _ in range(n_layers)]
    Omegas_init.append(rng.normal(0, np.sqrt(sigma2), size=(1, D_h)))  # single output

    h = rng.normal(size=(D_h, n_data))            # 100-D inputs ~ N(0, 1), one per column
    fs_init, var_f = [], []
    for Om in Omegas_init:                        # forward pass, biases zero
        fs_init.append(Om @ h)
        var_f.append(fs_init[-1].var())
        h = np.maximum(fs_init[-1], 0)

    dl_df = 2 * (fs_init[-1] - 0.0)               # least squares against y = 0
    var_g = [None] * n_layers
    for k in range(n_layers, 0, -1):              # backward pass
        dl_df = (fs_init[k - 1] > 0) * (Omegas_init[k].T @ dl_df)
        var_g[k - 1] = dl_df.var()

    layers = np.arange(n_layers)
    ratio = D_h * sigma2 / 2                      # predicted change per layer
    fwd = np.array(var_f[:n_layers]) / var_f[0]   # relative to layer 0
    bwd = np.array(var_g) / var_g[-1]             # relative to layer 49, where backprop starts
    line, = ax_f.semilogy(layers, fwd, label=f"$\\sigma^2_\\Omega$ = {sigma2}")
    ax_b.semilogy(layers, bwd, color=line.get_color())
    ax_f.semilogy(layers, ratio ** layers, "k:", lw=1)
    ax_b.semilogy(layers, ratio ** (n_layers - 1 - layers), "k:", lw=1)

ax_f.set(title="Forward: variance of $\\mathbf{f}_k$", xlabel="layer $k$",
         ylabel="variance relative to the starting layer")
ax_b.set(title="Backward: variance of $\\partial\\ell/\\partial\\mathbf{f}_k$", xlabel="layer $k$")
ax_f.set_yticks(10.0 ** np.arange(-80, 81, 40))
ax_f.legend()
plt.tight_layout()
plt.show()

## Aim: keep the variance the same between two layers

Drop the layer index and look at one block:

$$
\begin{aligned}
\mathbf{f}' &= \boldsymbol{\beta} + \boldsymbol{\Omega}\mathbf{h} \\
\mathbf{h} &= \mathbf{a}[\mathbf{f}]
\end{aligned}
$$

We want the variance of a unit $f'_i$ of the next layer's pre-activations,

$$
\sigma^2_{f'} = \mathbb{E}\bigl[(f_i' - \mathbb{E}[f_i'])^2\bigr],
$$

to equal the variance $\sigma^2_f$ of the current layer's. We need a few facts about expectations to compute it.

## Expectations: definition and rules

$$
\mathbb{E}\Bigl[\mathrm{g}[x]\Bigr] = \int \mathrm{g}[x]\, \mathrm{Pr}(x)\, dx
\qquad\approx\qquad
\frac{1}{N} \sum_{n=1}^{N} \mathrm{g}[x_n^*], \quad x_n^* \sim \mathrm{Pr}(x)
$$

The average value of $\mathrm{g}[x]$, weighted by how probable each $x$ is. Special cases (UDL Appendix C.2.1, eq. C.11): $\mathrm{g}[x] = x$ gives the mean $\mu$; $(x-\mu)^2$ the variance; $(x - \mu)^3$ the skew; $(x-\mu)^4$ the kurtosis.

Four rules (UDL Appendix C.2.1, eq. C.11):

$$
\begin{aligned}
&\text{Rule 1:} & \mathbb{E}\bigl[k\bigr] &= k \\
&\text{Rule 2:} & \mathbb{E}\bigl[k \cdot \mathrm{g}[x]\bigr] &= k \cdot \mathbb{E}\bigl[\mathrm{g}[x]\bigr] \\
&\text{Rule 3:} & \mathbb{E}\bigl[\mathrm{f}[x] + \mathrm{g}[x]\bigr] &= \mathbb{E}\bigl[\mathrm{f}[x]\bigr] + \mathbb{E}\bigl[\mathrm{g}[x]\bigr] \\
&\text{Rule 4:} & \mathbb{E}\bigl[\mathrm{f}[x]\mathrm{g}[y]\bigr] &= \mathbb{E}\bigl[\mathrm{f}[x]\bigr]\mathbb{E}\bigl[\mathrm{g}[y]\bigr] \qquad \text{if } x, y \text{ independent}
\end{aligned}
$$

## A useful identity: variance in terms of second moments

Claim: $\mathbb{E}\left[(x-\mu)^2\right] = \mathbb{E}[x^2] - \mathbb{E}[x]^2$, where $\mu = \mathbb{E}[x]$.


$$
\begin{aligned}
\mathbb{E}[(x-\mu)^2] &= \mathbb{E}[x^2 - 2x\mu + \mu^2] \\
&= \mathbb{E}[x^2] - \mathbb{E}[2x\mu] + \mathbb{E}[\mu^2] && \text{(Rule 3)} \\
&= \mathbb{E}[x^2] - 2\mu\mathbb{E}[x] + \mu^2 && \text{(Rules 1 and 2)} \\
&= \mathbb{E}[x^2] - 2\mu^2 + \mu^2 && \text{(Def'n of } \mu\text{)} \\
&= \mathbb{E}[x^2] - \mu^2 = \mathbb{E}[x^2] - \mathbb{E}[x]^2
\end{aligned}
$$


So $\sigma^2_{f'} = \mathbb{E}\bigl[f_i'^2\bigr] - \mathbb{E}[f_i']^2$. We compute the two terms in turn, starting with the mean.

## Mean of the pre-activations

$$
\mathbf{f}' = \boldsymbol{\beta} + \boldsymbol{\Omega}\mathbf{h}
\qquad\Longrightarrow\qquad
f_i' = \beta_i + \sum_{j=1}^{D_h} \Omega_{ij} h_j
$$


$$
\begin{aligned}
\mathbb{E}[f_i'] &= \mathbb{E}\left[\beta_i + \sum_{j=1}^{D_h} \Omega_{ij} h_j\right] \\
&= \mathbb{E}\left[\beta_i\right] + \sum_{j=1}^{D_h} \mathbb{E}\left[\Omega_{ij} h_j\right] && \text{(Rule 3)} \\
&= \mathbb{E}\left[\beta_i\right] + \sum_{j=1}^{D_h} \mathbb{E}\left[\Omega_{ij}\right] \mathbb{E}\left[h_j\right] && \text{(Rule 4: weights independent of activations)} \\
&= 0 + \sum_{j=1}^{D_h} 0 \cdot \mathbb{E}\left[h_j\right] = 0 && \text{(biases 0, weights mean 0)}
\end{aligned}
$$

So the mean of the pre-activations is zero and the variance reduces to the
second moment, $\sigma^2_{f'} = \mathbb{E}[f_i'^2]$.


## Variance of the pre-activations


With the initialization choices above (biases 0; weights mean 0, variance
$\sigma^2_\Omega$, independent of each other and of the activations):

$$
\begin{aligned}
\sigma^2_{f'} &= \mathbb{E}[f_i'^2] - \underbrace{\mathbb{E}[f_i']^2}_{0} \\
&= \mathbb{E}\left[\left(\beta_i + \sum_{j=1}^{D_h} \Omega_{ij} h_j\right)^2\right] \\
&= \mathbb{E}\left[\left(\sum_{j=1}^{D_h} \Omega_{ij} h_j\right)^2\right] && \text{(biases set to 0)} \\
&= \sum_{j=1}^{D_h} \mathbb{E}\left[\Omega_{ij}^2\right] \mathbb{E}\left[h_j^2\right] && \text{(cross terms vanish; Rule 4)} \\
&= \sum_{j=1}^{D_h} \sigma^2_\Omega \mathbb{E}\left[h_j^2\right] = \sigma^2_\Omega \sum_{j=1}^{D_h} \mathbb{E}\left[h_j^2\right]
\end{aligned}
$$

- Expanding the square gives $D_h$ squared terms and cross terms
  $\Omega_{ij}\Omega_{ik}h_j h_k$ with $j \neq k$; each cross term has
  expectation $\mathbb{E}[\Omega_{ij}]\mathbb{E}[\Omega_{ik}]\mathbb{E}[h_j h_k] = 0$
  because the weights are independent and zero mean.
- Because the $\Omega$'s are zero mean, $\mathbb{E}[\Omega_{ij}^2]$ is their variance.


## Variance with ReLU activations

The last factor depends on the activation function. For ReLU, with the previous layer's pre-activations $f_j$ zero mean and symmetric with variance $\sigma^2_f$:

$$
\begin{aligned}
\sigma^2_{f'} &= \sigma^2_\Omega \sum_{j=1}^{D_h} \mathbb{E}\left[h_j^2\right] = \sigma^2_\Omega \sum_{j=1}^{D_h} \mathbb{E}\left[\mathrm{ReLU}[f_j]^2\right] \\
&= \sigma^2_\Omega \sum_{j=1}^{D_h} \int_{-\infty}^{\infty} \mathrm{ReLU}[f_j]^2\, \mathrm{Pr}(f_j)\, df_j && \text{(definition of expectation)} \\
&= \sigma^2_\Omega \sum_{j=1}^{D_h} \int_{-\infty}^{\infty} (\mathbb{I}[f_j > 0] f_j)^2\, \mathrm{Pr}(f_j)\, df_j \\
&= \sigma^2_\Omega \sum_{j=1}^{D_h} \int_{0}^{\infty} f_j^2\, \mathrm{Pr}(f_j)\, df_j && \text{(ReLU zeroes the negative half)} \\
&= \sigma^2_\Omega \sum_{j=1}^{D_h} \frac{\sigma^2_f}{2} = \frac{D_h\, \sigma^2_\Omega\, \sigma^2_f}{2} && \text{(half the second moment, by symmetry)}
\end{aligned}
$$

The last step: for a zero-mean symmetric $f_j$, the integral over the positive half is half of $\mathbb{E}[f_j^2] = \sigma^2_f$.

A Monte Carlo check of both steps: sample pre-activations and one weight
matrix, apply the layer, and compare the empirical variance with the formula.
Change the three inputs.

In [ ]:
D_h, sigma2_omega, sigma2_f = 100, 0.05, 1.5
rng = np.random.default_rng(0)

f = rng.normal(0, np.sqrt(sigma2_f), size=(D_h, 10_000))      # previous layer, 10k examples
Omega = rng.normal(0, np.sqrt(sigma2_omega), size=(D_h, D_h))
f_next = Omega @ np.maximum(f, 0)                             # biases zero

print(f"E[ReLU[f]^2]: empirical {np.mean(np.maximum(f, 0) ** 2):.3f}, "
      f"predicted sigma2_f / 2 = {sigma2_f / 2:.3f}")
print(f"variance of f': empirical {f_next.var():.3f}, "
      f"predicted D_h sigma2_omega sigma2_f / 2 = {D_h * sigma2_omega * sigma2_f / 2:.3f}")

## Result: He (Kaiming) initialization

Since

$$
\sigma^2_{f'} = \frac{D_h\, \sigma^2_\Omega\, \sigma^2_f}{2},
$$

choosing

$$
\sigma^2_\Omega = \frac{2}{D_h}
$$

gives $\sigma^2_{f'} = \sigma^2_f$: the variance is the same in every layer, however deep the network.

This is **He initialization** or **Kaiming initialization**, from K. He, X. Zhang, S. Ren and J. Sun, "[Delving Deep into Rectifiers: Surpassing Human-Level Performance on ImageNet Classification](https://arxiv.org/abs/1502.01852)," ICCV 2015. The factor 2 is the ReLU correction; the earlier **Glorot (Xavier) initialization**, $\sigma^2_\Omega = 2/(D_h + D_{h'})$, was derived for symmetric activations like tanh ([Glorot and Bengio, 2010](https://proceedings.mlr.press/v9/glorot10a.html)).


![Kaiming He, <https://people.csail.mit.edu/kaiming/>](figs/05-Backpropagation-and-Initialization/kaiming-he.png)


## He initialization: forward (fan in) vs. backward (fan out)

- **Forward pass**: keep the variance of the activations in layer $k+1$ equal to that in layer $k$:

$$
\sigma^2_\Omega = \frac{2}{D_h} \qquad \longleftarrow \text{units in layer } k \text{ (fan in)}
$$

- **Backward pass**: the same derivation on $\boldsymbol{\Omega}_k^T$ gives $\sigma^2_{g} = D_{h'}\, \sigma^2_\Omega\, \sigma^2_{g'} / 2$, where $\sigma^2_{g'}$ and $\sigma^2_{g}$ are the variances of $\partial\ell/\partial\mathbf{f}$ in layers $k+1$ and $k$. Keeping them equal requires

$$
\sigma^2_\Omega = \frac{2}{D_{h'}} \qquad \longleftarrow \text{units in layer } k+1 \text{ (fan out)}
$$

- When the widths differ, pick one (PyTorch's default mode is `fan_in`) or use the average.
- In the experiment, $D_h = 100$ throughout, so $\sigma^2_\Omega = 2/100 = 0.02$: the one curve that stayed flat in both panels.


![UDL Fig. 7.7 again.](figs/05-Backpropagation-and-Initialization/weight-init-variance-forward-backward.png)


## What PyTorch does by default

- `nn.Linear` and `nn.Conv2d` initialize weights from a **uniform** distribution $\mathcal{U}(-\sqrt{k}, \sqrt{k})$ with $k = 1 / \text{fan\_in}$ ([`nn.Linear` docs](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html)), and biases from the same range. Internally that is `kaiming_uniform_(weight, a=math.sqrt(5))`.
- The variance of that uniform is $k/3 = 1/(3\, \text{fan\_in})$: **six times smaller** than He's $2/\text{fan\_in}$. On a plain 50-layer ReLU network the activation variance would shrink by a factor of 6 per layer. It is tolerable in practice because almost every modern network has normalization layers and residual connections (lecture 08), which stabilize the variance regardless of the initial scale.
- To get He initialization explicitly ([`nn.init` docs](https://docs.pytorch.org/docs/stable/nn.init.html#torch.nn.init.kaiming_normal_)):

In [ ]:
#| eval: false
#| code-fold: false
import torch.nn as nn

layer = nn.Linear(100, 100)
nn.init.kaiming_normal_(layer.weight, mode="fan_in", nonlinearity="relu")  # std = sqrt(2 / fan_in)
nn.init.zeros_(layer.bias)

`kaiming_normal_` samples $\mathcal{N}(0, \text{std}^2)$ with $\text{std} = \text{gain} / \sqrt{\text{fan\_mode}}$ and $\text{gain} = \sqrt{2}$ for ReLU, i.e. variance $2/D_h$. `mode="fan_out"` preserves the backward-pass variance instead.

Both claims in NumPy: the variance of PyTorch's default uniform against He's,
and what the default does to the activations of a plain ReLU stack (biases
zero here, to isolate the weights).

In [ ]:
fan_in, n_layers = 100, 5
rng = np.random.default_rng(0)

k = 1 / fan_in
W = rng.uniform(-np.sqrt(k), np.sqrt(k), size=(100_000, fan_in))
print(f"U(-sqrt(k), sqrt(k)) variance: empirical {W.var():.5f}, k/3 = {k / 3:.5f}")
print(f"He variance 2/fan_in = {2 / fan_in:.5f}; ratio = {(2 / fan_in) / W.var():.2f}")

f = rng.normal(size=(fan_in, 1000))
var_prev = f.var()
for layer in range(1, n_layers + 1):
    W = rng.uniform(-np.sqrt(k), np.sqrt(k), size=(fan_in, fan_in))
    f = W @ np.maximum(f, 0)
    print(f"layer {layer:2d}: variance {f.var():.2e}, shrank by {var_prev / f.var():.1f}x")
    var_prev = f.var()

## Initialization in practice: transformers

- GPT-2 initialized all weights from $\mathcal{N}(0, 0.02^2)$ and scaled the weights of the layers that write into the residual stream by $1/\sqrt{N}$, $N$ the number of residual layers, so that the variance accumulated along the residual path stays bounded with depth ([Radford et al., 2019](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf), section 2.3). The released [`model.py`](https://github.com/openai/gpt-2/blob/master/src/model.py) uses std 0.02 everywhere; the depth scaling is implemented in e.g. [nanoGPT](https://github.com/karpathy/nanoGPT/blob/master/model.py) as `0.02/sqrt(2 * n_layer)`.
- Many open training recipes keep a fixed small standard deviation rather than a width-dependent one, and rely on normalization layers for the rest. OLMo 2 initializes **every** parameter from $\mathcal{N}(0, 0.02^2)$, having dropped the depth-scaled initialization ($1/\sqrt{2 \cdot d_{\text{model}} \cdot \text{layer}}$ on output projections) of its predecessor because the simple scheme preserved activation and gradient scales better across layers ([OLMo et al., 2025](https://arxiv.org/abs/2501.00656), section 3.2).
- The derivation in this lecture is the one you can do by hand; residual connections and normalization change the bookkeeping but not the goal, a signal whose scale neither explodes nor vanishes across layers.

## What initialization does and does not guarantee

A good initialization keeps the **initial** gradients modestly sized, which keeps the first gradient-descent steps from swinging the parameters wildly. Smaller learning rates and learning-rate warmup (lecture 04) help with the same thing.

It gives **no guarantee** that:

- the model trains to a low loss,
- the activations or gradients stay well scaled once training moves the weights (the variance of the gradients starting equal does not mean it stays equal),
- the network has few inactive units; the He estimate *assumes* half the ReLUs are off.

If training misbehaves, look at the gradient norms per layer. And if you are reproducing a paper and training fails, check whether their code is public: papers often omit initialization details, especially when they claim the method is insensitive to them, and the code does not always match the description.

In fact, much of a trained network turns out to be unnecessary and can be pruned away, which leads to a striking observation about initialization.

# The Lottery Ticket Hypothesis

If most of a network can be pruned after training, which part mattered? The
Lottery Ticket Hypothesis says the answer depends on the initial values of the
surviving weights, not only on which weights survive.


## The Lottery Ticket Hypothesis

- **Hypothesis:** a randomly initialized dense network contains a subnetwork (a **"winning ticket"**) that is initialized such that, when trained in isolation, it matches the test accuracy of the full network in at most the same number of training iterations.
- **Finding it (iterative magnitude pruning):**
  1. Randomly initialize a dense network $\mathrm{f}[\mathbf{x}; \boldsymbol{\theta}_0]$.
  2. Train it to convergence, giving $\boldsymbol{\theta}_j$.
  3. Prune a percentage of the weights with the smallest magnitudes, creating a mask $\mathbf{m}$.
  4. **The critical step:** reset the remaining weights to their **original initial values** $\boldsymbol{\theta}_0$. The winning ticket is $\mathrm{f}[\mathbf{x}; \mathbf{m} \odot \boldsymbol{\theta}_0]$. Repeat.
- **Result:** subnetworks found this way are often **10–20% the size** of the original network yet learn faster and reach higher test accuracy.

J. Frankle and M. Carbin, "[The Lottery Ticket Hypothesis: Finding Sparse, Trainable Neural Networks](https://arxiv.org/abs/1803.03635)," ICLR 2019 (arXiv March 2018). The paper uses $\boldsymbol{\theta}$ for the parameters; we keep its notation here.

## Lottery tickets: what it says about initialization

- **Initialization matters, not just structure.** When a winning ticket's weights are **randomly re-initialized**, the same sparse structure trains slower and reaches lower accuracy than the dense network.
- **Generalization.** Winning tickets often show a smaller gap between training and test accuracy than the dense network.
- **Why over-parameterize?** The authors conjecture that larger dense networks are easier to train because they contain more candidate subnetworks, so one of them is more likely to be a winning ticket.
- **At scale.** On deeper networks (VGG-19, ResNet-18 on CIFAR-10) tickets were found only with **learning-rate warmup**; later work resets to an early training checkpoint rather than to $\boldsymbol{\theta}_0$.

J. Frankle and M. Carbin, "[The Lottery Ticket Hypothesis](https://arxiv.org/abs/1803.03635)," ICLR 2019.

# Summary

What to take away from this lecture; the recap deck and the knowledge check
are built from this list.


## Summary

- Training needs $\partial \ell_i / \partial \boldsymbol{\beta}_k$ and $\partial \ell_i / \partial \boldsymbol{\Omega}_k$ for every layer and every example; backpropagation computes them all in one forward and one backward pass.
- Forward pass: $\mathbf{f}_k = \boldsymbol{\beta}_k + \boldsymbol{\Omega}_k \mathbf{h}_k$, $\mathbf{h}_{k+1} = \mathbf{a}[\mathbf{f}_k]$; store everything.
- Backward pass: $\partial \ell_i / \partial \boldsymbol{\beta}_k = \partial \ell_i / \partial \mathbf{f}_k$; $\;\partial \ell_i / \partial \boldsymbol{\Omega}_k = (\partial \ell_i / \partial \mathbf{f}_k)\, \mathbf{h}_k^T$; $\;\partial \ell_i / \partial \mathbf{f}_{k-1} = \mathbb{I}[\mathbf{f}_{k-1} > 0] \odot (\boldsymbol{\Omega}_k^T\, \partial \ell_i / \partial \mathbf{f}_k)$.
- A derivative of a scalar has the shape of the thing you differentiate by; the linear link contributes $\boldsymbol{\Omega}^T$, the ReLU link an elementwise mask.
- Backward costs about twice forward ($6N$ FLOPs per token for an $N$-parameter transformer) and must keep all activations in memory; checkpointing trades recomputation for memory.
- Backpropagation is reverse-mode automatic differentiation on a computational graph: one backward pass gives the gradient of one scalar with respect to every parameter; derivatives along multiple paths add. Autograd builds the graph as the forward code runs and accumulates into `.grad`.
- With zero biases and independent zero-mean weights of variance $\sigma^2_\Omega$, a ReLU layer maps $\sigma^2_f \mapsto D_h \sigma^2_\Omega \sigma^2_f / 2$; He initialization $\sigma^2_\Omega = 2/D_h$ keeps it fixed (fan in for the forward pass, fan out for the backward pass).
- PyTorch's default `nn.Linear` init is uniform with variance $1/(3\,\text{fan\_in})$, not He; call `kaiming_normal_` for He. Transformers typically use a fixed small std with residual scaling.
- Initialization sets the starting point only; it guarantees nothing about training. Lottery tickets: a sparse subnetwork trains well *with its original initial values* and not with fresh ones.